# Transcribe Engine — Colab (T4 GPU)

Arabic YouTube → transcription dataset on **free Colab T4** (sm_75, same Turing
architecture as the local RTX 5000, so the `float16` defaults port exactly).

Pipeline: `yt-dlp → 16kHz wav → Silero VAD → Cohere-AR ASR → wav2vec2 align
→ QA → videos.jsonl → segments.parquet`.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Accept the gated model at
   [hf.co/CohereLabs/cohere-transcribe-arabic-07-2026](https://hf.co/CohereLabs/cohere-transcribe-arabic-07-2026).
3. Put your HF token in Colab Secrets as `HF_TOKEN` (or run `huggingface-cli login`).
4. Set `REPO_URL` below (or upload the repo zip).

In [ ]:
# @title 1. Check GPU + environment
!nvidia-smi
import torch

assert torch.cuda.is_available(), "No GPU! Runtime → Change runtime type → T4"
print("torch", torch.__version__, "| gpu", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
# @title 2. Install engine + clone repo
REPO_URL = ""  # @param {type:"string"}
assert REPO_URL, (
    "set REPO_URL to your git remote (or upload the repo and skip this cell)"
)
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q torch==2.11.0 --index-url https://download.pytorch.org/whl/cu128
!git clone -q $REPO_URL /content/transcribe-engine
%cd /content/transcribe-engine
!pip install -q -e ".[engine]"
print("installed. coherex + yt-dlp + polars ready.")

In [ ]:
# @title 3. HF token for the gated Cohere model
import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF token loaded from Colab secrets")
except Exception:
    print("No HF_TOKEN in secrets — run:  huggingface-cli login")

# models are cached in the session; a re-run without re-download needs a Drive
# cache (see cell 6) or a persistent runtime.

In [ ]:
# @title 4. Transcribe one video (or a playlist)
VIDEO_URL = "https://www.youtube.com/watch?v=ZXWzWWXzPmU"  # @param {type:"string"}

# bare watch?v= URLs only for single videos (a &list= param makes yt-dlp
# expand the whole playlist; the engine has a guard but be explicit)
!python -m src.cli video "$VIDEO_URL" \
  --to-parquet data/output/segments.parquet

# playlist (checkpointed, resumable — safe to re-run after disconnect):
# !python -m src.cli playlist "https://youtube.com/playlist?list=PLxxx" \
#   --max-videos 5 --to-parquet data/output/segments.parquet

In [ ]:
# @title 5. Inspect the result
import json, polars as pl

rec = json.loads(open("data/output/videos.jsonl", encoding="utf-8").readline())
print(
    rec["video_title"],
    "|",
    len(rec["segments"]),
    "segments |",
    len(rec["word_segments"]),
    "words | qa:",
    rec.get("qa") or "clean",
)
print("qa_scores:", rec.get("qa_scores"))
df = pl.read_parquet("data/output/segments.parquet")
print(df.shape, df.columns)
df.head(10)

In [ ]:
# @title 6. Persist outputs to Google Drive
PERSIST = False  # @param {type:"boolean"}
if PERSIST:
    from google.colab import drive

    drive.mount("/content/drive")
    !mkdir -p "/content/drive/MyDrive/transcribe-engine/output"
    !cp -r data/output/* "/content/drive/MyDrive/transcribe-engine/output/"
    print("copied to Drive/transcribe-engine/output")
else:
    print("skipped — outputs die with the runtime; enable to keep them")

## Notes

- **T4 = sm_75**, same as the local RTX 5000: `--compute-type float16`
  is already the default; never use `bfloat16` on Turing.
- **VRAM**: ASR ~4 GB + aligner ~2 GB resident; T4's 16 GB is the same
  budget as local. Batch auto-drops 8 → 4 on OOM.
- **Throttled links**: audio downloads and the one-time 4 GB model fetch
  can be slow; the checkpoint (`--db`) makes re-runs resume for free.
- **Text policy**: `transcript_raw` verbatim + `transcript_norm` folded
  (`norm_version` stamped). Audio is deleted after transcription by default.
- **Diarization** (`--diarize`) runs ungated Nemotron-3 — no model
  acceptance needed (pyannote was removed, ADR 18).
- Full reference: `README.md` in the repo.